# A Practical Guide to Measuring Text Similarity

### A hands-on look at how text can be compared by words, structure, and meaning, and how these NLP techniques can be used to evaluate variation in AI-generated answers.

**Joe Domaleski**  
Marketing Data Science

---

## Introduction

Two pieces of text can look different while saying essentially the same thing. They can also look almost identical while making very different claims.

That creates an important measurement problem. If we want to compare customer reviews, survey responses, social media posts, documents, or answers generated by an AI system, what does it actually mean for two texts to be "similar"?

There is no single best similarity score. Different methods measure different things.

In this notebook, we will compare several approaches:

1. Exact string matching
2. Levenshtein similarity
3. Jaccard similarity
4. TF-IDF with cosine similarity
5. Sentence embeddings with cosine similarity

We will start with controlled examples where we know what changed. Then we will use the same methods to compare multiple AI-generated answers to the same marketing question.

The goal is not to declare one method the winner. The goal is to understand what each method sees, what it misses, and which method makes sense for the question we are trying to answer.

## 1. Setup

This notebook is designed to run in **Google Colab**.

Most of the libraries we need are already available in Colab. We will install two additional packages:

- `rapidfuzz` for normalized Levenshtein similarity
- `sentence-transformers` for semantic embeddings

The first time the embedding model runs, Colab will download the pretrained model from Hugging Face.

In [ ]:
!pip -q install rapidfuzz sentence-transformers

### Import the libraries

We will use:

- `pandas` and `numpy` for data handling
- `matplotlib` for visualization
- `scikit-learn` for TF-IDF and cosine similarity
- `rapidfuzz` for Levenshtein similarity
- `sentence-transformers` for semantic embeddings

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from itertools import combinations
from rapidfuzz.distance import Levenshtein
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

pd.set_option("display.max_colwidth", 120)

print("Libraries loaded successfully.")

## 2. Controlled Text Examples

Before comparing a collection of AI answers, it helps to start with examples where we deliberately control the differences.

The examples below are designed to test several situations:

- identical text
- similar meaning with different wording
- high word overlap with opposite meaning
- similar concepts expressed differently
- nearly identical wording with a materially different number
- unrelated text

These examples will help us see why a high similarity score does not always mean two texts agree.

In [ ]:
text_pairs = [
    {
        "pair": "A",
        "description": "Exact match",
        "text_1": "Marketing attribution identifies which channels contribute to a conversion.",
        "text_2": "Marketing attribution identifies which channels contribute to a conversion."
    },
    {
        "pair": "B",
        "description": "Same idea, different wording",
        "text_1": "Marketing attribution identifies which channels contribute to a conversion.",
        "text_2": "Marketing attribution helps determine which marketing channels influenced a customer conversion."
    },
    {
        "pair": "C",
        "description": "Nearly identical wording, opposite direction",
        "text_1": "The campaign increased conversions by 15 percent.",
        "text_2": "The campaign decreased conversions by 15 percent."
    },
    {
        "pair": "D",
        "description": "Related concept, different wording",
        "text_1": "Customer retention measures how well a business keeps existing customers.",
        "text_2": "A strong retention rate means customers continue doing business with the company."
    },
    {
        "pair": "E",
        "description": "Same wording, materially different number",
        "text_1": "Revenue increased by 10 percent last quarter.",
        "text_2": "Revenue increased by 100 percent last quarter."
    },
    {
        "pair": "F",
        "description": "Unrelated topics",
        "text_1": "Email open rates can help marketers evaluate subject line performance.",
        "text_2": "Linear regression estimates the relationship between variables."
    }
]

pairs_df = pd.DataFrame(text_pairs)
pairs_df[["pair", "description", "text_1", "text_2"]]

## 3. Exact Match

The simplest possible comparison is an exact match.

Two texts receive a score of `1` only if every character is identical. Otherwise, they receive `0`.

This is useful when exact duplication matters, such as identifying repeated records or verifying that a generated response is exactly the same as a previous one.

It is not useful for measuring meaning. A single punctuation change makes two texts different even if a human reader would consider them equivalent.

In [ ]:
def exact_match(text_1, text_2):
    return int(text_1 == text_2)

pairs_df["exact_match"] = pairs_df.apply(
    lambda row: exact_match(row["text_1"], row["text_2"]),
    axis=1
)

pairs_df[["pair", "description", "exact_match"]]

## 4. Levenshtein Similarity

Levenshtein distance measures how many single-character edits are needed to turn one string into another.

Those edits can be:

- insertions
- deletions
- substitutions

A raw edit distance can be difficult to compare across texts of different lengths, so we will use a **normalized Levenshtein similarity score** from 0 to 1.

- `1.00` means the strings are identical
- scores closer to `0.00` mean more character-level editing would be required

This method is good at measuring structural or spelling similarity, but it does not understand meaning.

In [ ]:
def levenshtein_similarity(text_1, text_2):
    return Levenshtein.normalized_similarity(text_1, text_2)

pairs_df["levenshtein"] = pairs_df.apply(
    lambda row: levenshtein_similarity(row["text_1"], row["text_2"]),
    axis=1
)

pairs_df[["pair", "description", "levenshtein"]].round(3)

## 5. Jaccard Similarity

Jaccard similarity compares the overlap between two sets.

For text, we can turn each sentence into a set of normalized words and ask:

> What proportion of the unique words found in either text appear in both?

The formula is:

$$
J(A,B) = \frac{|A \cap B|}{|A \cup B|}
$$

A score of `1` means both texts contain the same set of words. A score of `0` means they share no words.

Jaccard similarity ignores word order and frequency. That makes it easy to understand, but it also means important words can be treated the same as unimportant ones.

In [ ]:
def tokenize_words(text):
    return set(re.findall(r"\b\w+\b", text.lower()))

def jaccard_similarity(text_1, text_2):
    words_1 = tokenize_words(text_1)
    words_2 = tokenize_words(text_2)

    union = words_1 | words_2
    if not union:
        return 1.0

    intersection = words_1 & words_2
    return len(intersection) / len(union)

pairs_df["jaccard"] = pairs_df.apply(
    lambda row: jaccard_similarity(row["text_1"], row["text_2"]),
    axis=1
)

pairs_df[["pair", "description", "jaccard"]].round(3)

## 6. TF-IDF with Cosine Similarity

Jaccard similarity treats every word equally. TF-IDF improves on that by giving more weight to words that are informative within a collection of documents and less weight to words that appear frequently.

TF-IDF stands for:

- **Term Frequency**: how often a term appears in a document
- **Inverse Document Frequency**: how uncommon that term is across the documents being analyzed

Once each text is represented as a TF-IDF vector, we can compare the angle between those vectors using **cosine similarity**.

Cosine similarity ranges from 0 to 1 in this application:

- values near `1` indicate similar term usage
- values near `0` indicate little overlap in the weighted vocabulary

This is a major step beyond simple word matching, but it still depends heavily on the words that appear in the text. It does not fully understand semantic meaning.

In [ ]:
def tfidf_cosine_similarity(text_1, text_2):
    vectorizer = TfidfVectorizer()
    matrix = vectorizer.fit_transform([text_1, text_2])
    return cosine_similarity(matrix[0:1], matrix[1:2])[0, 0]

pairs_df["tfidf_cosine"] = pairs_df.apply(
    lambda row: tfidf_cosine_similarity(row["text_1"], row["text_2"]),
    axis=1
)

pairs_df[["pair", "description", "tfidf_cosine"]].round(3)

## 7. Sentence Embeddings with Cosine Similarity

Modern NLP gives us another way to compare text: **embeddings**.

An embedding converts a sentence into a numeric vector designed to capture patterns associated with semantic meaning. Sentences that express similar ideas often end up closer together in the embedding space, even when they use different words.

We will use the pretrained `all-MiniLM-L6-v2` model from the Sentence Transformers project.

After creating an embedding for each sentence, we will again use cosine similarity to compare the vectors.

This moves us from primarily **lexical similarity** toward a model-based estimate of **semantic similarity**. It can recognize paraphrases much better than simple word-overlap methods, but it should not be treated as proof that two statements fully agree or are both correct.

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

def embedding_cosine_similarity(text_1, text_2):
    embeddings = model.encode(
        [text_1, text_2],
        normalize_embeddings=True
    )
    return float(np.dot(embeddings[0], embeddings[1]))

pairs_df["embedding_cosine"] = pairs_df.apply(
    lambda row: embedding_cosine_similarity(row["text_1"], row["text_2"]),
    axis=1
)

pairs_df[["pair", "description", "embedding_cosine"]].round(3)

## 8. Compare the Methods Side by Side

Now we can put all of the scores in one table.

This is where the differences between methods become easier to see.

Pay particular attention to **Pair C**:

> The campaign increased conversions by 15 percent.  
> The campaign decreased conversions by 15 percent.

Almost every word is identical, so several similarity measures may score the pair highly. But the business meaning changes dramatically because one word reverses the conclusion.

Pair E introduces a related problem with numbers:

> Revenue increased by 10 percent last quarter.  
> Revenue increased by 100 percent last quarter.

The wording is nearly identical, yet the magnitude of the business result is dramatically different.

These examples highlight an important limitation to remember throughout this notebook:

> **Similarity is not the same thing as factual agreement, logical agreement, numerical equivalence, or correctness.**

In [ ]:
comparison_columns = [
    "pair",
    "description",
    "exact_match",
    "levenshtein",
    "jaccard",
    "tfidf_cosine",
    "embedding_cosine"
]

comparison_df = pairs_df[comparison_columns].copy()

numeric_cols = [
    "levenshtein",
    "jaccard",
    "tfidf_cosine",
    "embedding_cosine"
]

comparison_df[numeric_cols] = comparison_df[numeric_cols].round(3)
comparison_df

### Visual comparison of the similarity scores

A chart makes it easier to see how the methods react differently to the same text pairs.

The exact-match score is omitted from this chart because it is binary. We will compare the four graded similarity measures.

In [ ]:
plot_df = comparison_df.set_index("pair")[
    ["levenshtein", "jaccard", "tfidf_cosine", "embedding_cosine"]
]

ax = plot_df.plot(
    kind="bar",
    figsize=(11, 6)
)

ax.set_title("Text Similarity Scores by Method")
ax.set_xlabel("Text Pair")
ax.set_ylabel("Similarity Score")
ax.set_ylim(0, 1.05)
ax.legend(title="Method", bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()

## 9. Moving from Two Texts to AI-Generated Answers

Comparing two sentences is useful for learning the mechanics, but the more interesting use case is comparing many answers generated from the same prompt.

Suppose we ask an AI system:

> **What is marketing attribution, and why does it matter?**

Below are eight example AI-generated answers created for this notebook. They intentionally vary in wording, length, detail, and emphasis.

Most communicate broadly similar ideas. One answer also makes a questionable overstatement so we can see whether a similarity score alone is enough to identify a substantive problem.

In a future experiment, these responses could come directly from repeated API calls rather than being stored in the notebook.

In [ ]:
prompt = "What is marketing attribution, and why does it matter?"

ai_answers = {
    "Answer 1": (
        "Marketing attribution is the process of identifying which marketing touchpoints "
        "contributed to a conversion. It matters because it helps marketers understand "
        "which channels and campaigns are influencing results so they can make better "
        "budget and strategy decisions."
    ),
    "Answer 2": (
        "Marketing attribution helps determine which ads, channels, and customer interactions "
        "played a role in producing a conversion. By connecting marketing activity to outcomes, "
        "businesses can evaluate performance and allocate resources more effectively."
    ),
    "Answer 3": (
        "Attribution is a way to assign credit for a sale or conversion across the marketing "
        "touchpoints a customer encountered. It is useful because marketers can see which "
        "parts of the customer journey appear to contribute most to business results."
    ),
    "Answer 4": (
        "Marketing attribution connects customer conversions back to the marketing interactions "
        "that preceded them. The goal is to understand what influenced the outcome and use that "
        "information to improve campaigns, channel mix, and spending decisions."
    ),
    "Answer 5": (
        "Marketing attribution measures the relationship between marketing touchpoints and "
        "conversions. It matters because marketers rarely have unlimited budgets, so they need "
        "evidence about which activities deserve more investment and which may deserve less."
    ),
    "Answer 6": (
        "Attribution tries to answer a practical question: which marketing efforts helped produce "
        "this result? Different attribution models divide credit differently, but all are attempts "
        "to connect marketing exposure with outcomes and improve decision-making."
    ),
    "Answer 7": (
        "Marketing attribution is the practice of tracing conversions to the channels or interactions "
        "that influenced them. It can help teams compare marketing performance, understand customer "
        "journeys, and make more informed decisions about future campaigns."
    ),
    "Answer 8": (
        "Marketing attribution identifies the single marketing channel that caused a conversion. "
        "It matters because the last interaction before the sale should receive all of the credit, "
        "making last-click attribution the most accurate method for every business."
    )
}

answers_df = pd.DataFrame(
    [{"answer": name, "text": text} for name, text in ai_answers.items()]
)

print("Prompt:")
print(prompt)
print("\nAI-generated answers:")
answers_df

## 10. Compare Each AI Answer to a Reference Answer

One common evaluation strategy is to compare every response with a reference.

For demonstration purposes, we will use **Answer 1** as the reference. This does not mean it is the objectively correct or ideal answer. It simply gives us a consistent baseline.

We will calculate:

- Levenshtein similarity
- Jaccard similarity
- TF-IDF cosine similarity
- embedding cosine similarity

This lets us see whether an answer can be lexically different but semantically similar, or lexically similar while making a different substantive claim.

In [ ]:
reference_name = "Answer 1"
reference_text = ai_answers[reference_name]

reference_results = []

for answer_name, answer_text in ai_answers.items():
    reference_results.append({
        "answer": answer_name,
        "levenshtein": levenshtein_similarity(reference_text, answer_text),
        "jaccard": jaccard_similarity(reference_text, answer_text),
        "tfidf_cosine": tfidf_cosine_similarity(reference_text, answer_text),
        "embedding_cosine": embedding_cosine_similarity(reference_text, answer_text)
    })

reference_df = pd.DataFrame(reference_results).round(3)
reference_df

### A note about the problematic answer

Answer 8 contains many of the same marketing-attribution terms as the other answers, but it makes a much stronger claim:

> "...the last interaction before the sale should receive all of the credit..."

A text-similarity model may still consider that answer similar because it is about the same topic and uses much of the same vocabulary.

This demonstrates an important distinction:

- **Text similarity** asks whether two texts resemble each other.
- **Factual evaluation** asks whether the claims are supported.
- **Logical evaluation** asks whether the reasoning is sound.
- **Quality evaluation** asks whether the answer is useful for the intended purpose.

Those are related questions, but they are not interchangeable.

## 11. Pairwise Semantic Similarity Across All AI Answers

When we have many responses, comparing everything to a single reference can hide useful information.

Instead, we can compare every answer with every other answer.

For eight answers, that produces an 8 × 8 similarity matrix. The diagonal will always equal 1 because every answer is identical to itself.

Here we will use sentence embeddings because our main interest is semantic similarity.

In [ ]:
answer_names = list(ai_answers.keys())
answer_texts = list(ai_answers.values())

answer_embeddings = model.encode(
    answer_texts,
    normalize_embeddings=True
)

embedding_matrix = cosine_similarity(answer_embeddings)

embedding_similarity_df = pd.DataFrame(
    embedding_matrix,
    index=answer_names,
    columns=answer_names
).round(3)

embedding_similarity_df

## 12. Visualize the Similarity Matrix

A heatmap makes the pairwise comparison easier to scan.

Values closer to 1 indicate greater semantic similarity according to the embedding model.

The heatmap is useful when an experiment contains enough responses that reading every pair manually becomes impractical.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

image = ax.imshow(
    embedding_similarity_df.values,
    vmin=0,
    vmax=1
)

ax.set_xticks(range(len(answer_names)))
ax.set_yticks(range(len(answer_names)))
ax.set_xticklabels(answer_names, rotation=45, ha="right")
ax.set_yticklabels(answer_names)

for i in range(len(answer_names)):
    for j in range(len(answer_names)):
        ax.text(
            j,
            i,
            f"{embedding_similarity_df.iloc[i, j]:.2f}",
            ha="center",
            va="center"
        )

ax.set_title("Pairwise Semantic Similarity of AI-Generated Answers")
fig.colorbar(image, ax=ax, label="Cosine Similarity")

plt.tight_layout()
plt.show()

## 13. Find the Most and Least Similar Answer Pairs

A matrix is useful visually, but we may also want direct summary statistics.

We will calculate every unique answer pair and identify:

- the most semantically similar pair
- the least semantically similar pair

With eight answers there are only 28 unique pairs. With hundreds of answers there can be tens of thousands of pairwise comparisons, so automating this step becomes increasingly useful.

In [ ]:
pairwise_results = []

for i, j in combinations(range(len(answer_names)), 2):
    pairwise_results.append({
        "answer_1": answer_names[i],
        "answer_2": answer_names[j],
        "embedding_similarity": embedding_matrix[i, j]
    })

pairwise_df = pd.DataFrame(pairwise_results).sort_values(
    "embedding_similarity",
    ascending=False
).reset_index(drop=True)

pairwise_df["embedding_similarity"] = pairwise_df["embedding_similarity"].round(3)

print("Most similar pair:")
display(pairwise_df.head(1))

print("\nLeast similar pair:")
display(pairwise_df.tail(1))

print("\nAll unique pairs:")
pairwise_df

## 14. Which Answer Is Most Representative?

We can also ask which response is most similar, on average, to all of the other responses.

One simple approach is to calculate the mean pairwise semantic similarity for each answer, excluding its similarity with itself.

The response with the highest average similarity can be thought of as the most **central** or **representative** answer in this particular set.

This does **not** mean it is the best or most accurate answer. It only means it is closest to the center of what the group is saying.

In [ ]:
average_similarities = []

for i, answer_name in enumerate(answer_names):
    other_scores = np.delete(embedding_matrix[i], i)

    average_similarities.append({
        "answer": answer_name,
        "average_similarity_to_others": other_scores.mean()
    })

centrality_df = pd.DataFrame(average_similarities).sort_values(
    "average_similarity_to_others",
    ascending=False
).reset_index(drop=True)

centrality_df["average_similarity_to_others"] = (
    centrality_df["average_similarity_to_others"].round(3)
)

centrality_df

## 15. Compare Multiple Similarity Methods Across the AI Answers

Semantic embeddings are powerful, but it is useful to see how different methods behave on the same collection.

The next table summarizes the average similarity across every unique pair of AI answers for four methods.

This gives us a compact way to compare how strict or forgiving each method is on the same data.

In [ ]:
method_pair_scores = []

for answer_1, answer_2 in combinations(answer_names, 2):
    text_1 = ai_answers[answer_1]
    text_2 = ai_answers[answer_2]

    method_pair_scores.append({
        "answer_1": answer_1,
        "answer_2": answer_2,
        "levenshtein": levenshtein_similarity(text_1, text_2),
        "jaccard": jaccard_similarity(text_1, text_2),
        "tfidf_cosine": tfidf_cosine_similarity(text_1, text_2),
        "embedding_cosine": embedding_cosine_similarity(text_1, text_2)
    })

all_methods_df = pd.DataFrame(method_pair_scores)

method_summary_df = (
    all_methods_df[
        ["levenshtein", "jaccard", "tfidf_cosine", "embedding_cosine"]
    ]
    .agg(["mean", "min", "max"])
    .T
    .round(3)
)

method_summary_df

## 16. The Contradiction Trap

One of the easiest mistakes to make with similarity metrics is assuming that a high score means two statements agree.

Consider these sentences again:

- **The campaign increased conversions by 15 percent.**
- **The campaign decreased conversions by 15 percent.**

They have nearly identical structure and vocabulary. Only one word changes, but that one word reverses the business conclusion.

Let's isolate this example and look at the scores again.

In [ ]:
contradiction_1 = "The campaign increased conversions by 15 percent."
contradiction_2 = "The campaign decreased conversions by 15 percent."

contradiction_results = pd.DataFrame([{
    "levenshtein": levenshtein_similarity(contradiction_1, contradiction_2),
    "jaccard": jaccard_similarity(contradiction_1, contradiction_2),
    "tfidf_cosine": tfidf_cosine_similarity(contradiction_1, contradiction_2),
    "embedding_cosine": embedding_cosine_similarity(contradiction_1, contradiction_2)
}]).round(3)

contradiction_results

The lesson is straightforward:

> **Similarity measures are measurements, not judgment.**

A high score can tell us that two texts share structure, vocabulary, or semantic context. It cannot automatically tell us that both are true, that they agree on every important detail, or that they are equally useful.

For AI evaluation, similarity analysis should usually be one part of a broader evaluation process.

### A second trap: numbers can change the business meaning

Text-similarity methods can also miss the importance of a numerical change.

Compare:

- **Revenue increased by 10 percent last quarter.**
- **Revenue increased by 100 percent last quarter.**

Only one character changes, but the second claim describes a result ten times as large.

For marketing and analytics work, that distinction can be more important than the overall textual similarity. A high similarity score should therefore not be interpreted as evidence that two responses are numerically equivalent.

In [ ]:
numeric_1 = "Revenue increased by 10 percent last quarter."
numeric_2 = "Revenue increased by 100 percent last quarter."

numeric_difference_results = pd.DataFrame([{
    "levenshtein": levenshtein_similarity(numeric_1, numeric_2),
    "jaccard": jaccard_similarity(numeric_1, numeric_2),
    "tfidf_cosine": tfidf_cosine_similarity(numeric_1, numeric_2),
    "embedding_cosine": embedding_cosine_similarity(numeric_1, numeric_2)
}]).round(3)

numeric_difference_results

## 17. Scaling the Approach to Future AI API Experiments

The examples above use eight stored responses so the notebook is reproducible and does not require an API key.

The same workflow can be extended to a much larger experiment:

1. Send the same prompt to an AI model many times.
2. Store each response with metadata such as model, timestamp, temperature, and run number.
3. Create sentence embeddings for all responses.
4. Compute a pairwise similarity matrix.
5. Measure the distribution of similarity scores.
6. Identify clusters, unusual responses, and representative responses.
7. Compare similarity across models, prompts, or parameter settings.

For example, 100 AI responses produce:

$$
\frac{100 \times 99}{2} = 4,950
$$

unique response pairs.

At that scale, automated similarity analysis becomes much more practical than manually reading every pair.

In [ ]:
def analyze_response_collection(responses, model):
    # Create embeddings and a cosine-similarity matrix for a dictionary
    # of named text responses.
    names = list(responses.keys())
    texts = list(responses.values())

    embeddings = model.encode(
        texts,
        normalize_embeddings=True
    )

    matrix = cosine_similarity(embeddings)

    similarity_df = pd.DataFrame(
        matrix,
        index=names,
        columns=names
    )

    return similarity_df


scaled_example = analyze_response_collection(ai_answers, model)
scaled_example.round(3)

## 18. Optional Template for Your Own AI Responses

To analyze your own results later, replace the example dictionary below with responses collected from an API, CSV file, spreadsheet, or another source.

The rest of the analysis can stay essentially the same.

In [ ]:
my_responses = {
    "Run 1": "Paste or load the first AI response here.",
    "Run 2": "Paste or load the second AI response here.",
    "Run 3": "Paste or load the third AI response here."
}

# Uncomment after replacing the placeholder text:
# my_similarity_matrix = analyze_response_collection(my_responses, model)
# display(my_similarity_matrix.round(3))

## 19. Key Takeaways

There is no universal text-similarity score.

Each method answers a somewhat different question:

| Method | What It Mostly Measures | Useful For | Important Limitation |
|---|---|---|---|
| Exact match | Character-for-character equality | Duplicate detection, reproducibility | Any change makes the texts different |
| Levenshtein | Character-level editing distance | Typos, revisions, near-duplicate strings | Does not understand meaning |
| Jaccard | Shared unique words | Simple vocabulary overlap | Ignores order and importance |
| TF-IDF + cosine | Weighted term overlap | Document similarity, search, classical NLP | Still tied closely to vocabulary |
| Embeddings + cosine | Semantic proximity | Paraphrases, AI responses, meaning-level comparison | Similarity does not guarantee factual agreement |

For comparing AI-generated answers, embeddings are especially useful because different responses can express similar ideas using very different words.

But the contradiction example demonstrates why no similarity metric should be treated as a truth detector.

A useful AI-evaluation framework may eventually combine several dimensions:

- semantic similarity
- factual consistency
- numerical consistency
- presence of required facts
- citation or source agreement
- length and structure
- tone
- outlier detection

That turns text similarity from an abstract NLP concept into a practical tool for evaluating variation across many AI-generated answers.

## 20. Next Steps

This notebook provides the foundation for a larger experiment.

A future project could repeatedly call an AI API with the same prompt and measure how much the answers vary across dozens, hundreds, or thousands of runs.

Possible extensions include:

- comparing different models
- changing temperature or other generation settings
- clustering semantically similar answers
- visualizing embeddings with PCA or UMAP
- tracking whether key facts appear consistently
- identifying outlier or anomalous responses
- comparing semantic similarity with human ratings

The important idea is that AI variation can be measured. The wording may change from one response to another, but NLP gives us several ways to quantify how much actually changed.

## Resources

Alammar, Jay, and Maarten Grootendorst. *Hands-On Large Language Models: Language Understanding and Generation*. O’Reilly Media, 2024. https://www.oreilly.com/library/view/hands-on-large-language/9781098150952/

Domaleski, Joe. “A Marketer’s Guide to NLP: How Machines Actually Process and Understand Language.” *Marketing Data Science*, 19 Oct. 2025. https://blog.marketingdatascience.ai/a-marketers-guide-to-nlp-how-machines-actually-process-and-understand-language-3d452febb3de

Domaleski, Joe. “Sentiment Analysis of Online Reviews Using R.” *Marketing Data Science*, 22 Sept. 2024. https://blog.marketingdatascience.ai/sentiment-analysis-of-online-reviews-using-r-e2afbc9fcc68

Domaleski, Joe. “UBCF vs. IBCF: Comparing Marketing Recommendation System Algorithms in R.” *Marketing Data Science*, 6 Apr. 2025. https://blog.marketingdatascience.ai/ubcf-vs-ibcf-comparing-marketing-recommendation-system-algorithms-in-r-38ff36bf05d3

Kocaman, Ahmet Münir. “How to Measure Text Similarity: A Comprehensive Guide.” *Medium*, 7 Oct. 2023. https://medium.com/@ahmetmnirkocaman/how-to-measure-text-similarity-a-comprehensive-guide-6c6f24fc01fe

Ladd, John R. “Understanding and Using Common Similarity Measures for Text Analysis.” *Programming Historian*, no. 9, 5 May 2020. https://doi.org/10.46430/phen0089

Levy, Avivit, B. Riva Shalom, and Michal Chalamish. “A Guide to Similarity Measures.” *arXiv*, 7 Aug. 2024. https://arxiv.org/abs/2408.07706

Reimers, Nils, and Iryna Gurevych. “all-MiniLM-L6-v2.” *Hugging Face*, 2021. https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

Vaswani, Ashish, et al. “Attention Is All You Need.” *Advances in Neural Information Processing Systems*, vol. 30, 2017, pp. 5998–6008. https://arxiv.org/abs/1706.03762

Wang, Wenhui, et al. “MiniLM: Deep Self-Attention Distillation for Task-Agnostic Compression of Pre-Trained Transformers.” *arXiv*, 2020. https://arxiv.org/abs/2002.10957